# Residual STL (robust)
        **timeseries › anomaly** · notebook `stl_residual`

        **Kapan dipakai:** Musiman kuat: pisahkan tren dan musiman, lalu cari sisa yang aneh.

        **Jebakan umum:** Satu periode musiman saja (di sini harian); musiman mingguan masih tersisa.

**Data demo:** NYC taxi (NAB), 30 menit sekali, 5 jendela anomali nyata berlabel (maraton NYC, Thanksgiving, Natal, Tahun Baru, badai salju). Anomali tidak dipakai saat melatih.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
PATH = ROOT / "data_latihan/timeseries/anomaly/nyc_taxi.csv"
P = 48                 # panjang siklus harian (48 titik x 30 menit)
W = 48                            # lebar jendela untuk metode berbasis jendela
QUANTILE = 0.995                  # ambang alarm = kuantil skor
MODEL_KEY = "stl_residual"
OUT = out_dir("timeseries", "anomaly", MODEL_KEY)
set_seed(42)
from superdb.timeseries import event_metrics, label_windows

## 2. Data dan jendela anomali berlabel

In [ ]:
df = pd.read_csv(PATH, parse_dates=["timestamp"]); s = df.set_index("timestamp")["value"].astype(float).asfreq("30min").interpolate()
labels = df.set_index("timestamp")["is_anomaly"].reindex(s.index).fillna(0).astype(int).to_numpy(); wins = label_windows(labels)
print(len(s), "titik |", len(wins), "jendela anomali:", [(str(s.index[a].date()), str(s.index[b].date())) for a, b in wins])
fig, ax = plt.subplots(figsize=(11, 3)); ax.plot(s.index, s.values, lw=.5)
for a, b in wins: ax.axvspan(s.index[a], s.index[b], color="r", alpha=.2)
ax.set_title("Data dengan jendela anomali (merah)"); plt.tight_layout(); plt.show()

## 3. Skor anomali
Semua metode berbentuk `score(s)` yang mengembalikan satu skor per titik (makin besar makin aneh).

In [ ]:
import numpy as np, pandas as pd
from superdb.timeseries import window_matrix, to_point_scores
from statsmodels.tsa.seasonal import STL

def score(s):
    res = STL(s, period=P, robust=True).fit().resid
    mad = (res - res.median()).abs().median() * 1.4826
    return (res.abs() / mad).to_numpy()
import time; t0 = time.time(); sc = np.asarray(score(s), float); print(f"waktu: {time.time() - t0:.1f} dtk")
assert sc.shape == (len(s),) and np.isfinite(sc).all()

## 4. Evaluasi
- **recall_jendela**: berapa jendela anomali yang memuat setidaknya satu alarm.
- **alarm_klaster_di_luar**: false alarm (titik berurutan digabung jadi satu klaster).
- **pr_auc_titik**: kasar, karena jendela label panjang dan tidak semua titik di dalamnya aneh.

In [ ]:
rows = {f"q={q}": event_metrics(sc, labels, wins, q) for q in (0.98, 0.99, 0.995, 0.999)}
res = pd.DataFrame(rows).T; display(res.round(4))
m = event_metrics(sc, labels, wins, QUANTILE); display(show_table(m, f"ringkasan pada kuantil {QUANTILE}"))

In [ ]:
thr = np.quantile(sc, QUANTILE)
fig, ax = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
ax[0].plot(s.index, s.values, lw=.5); ax[0].scatter(s.index[sc >= thr], s.values[sc >= thr], c="r", s=6, label="alarm"); ax[0].legend()
for a, b in wins: ax[0].axvspan(s.index[a], s.index[b], color="orange", alpha=.2)
ax[1].plot(s.index, sc, lw=.5); ax[1].axhline(thr, color="r", ls="--"); ax[1].set_title("skor anomali")
plt.tight_layout(); plt.savefig(OUT / "anomaly.png", dpi=130); plt.show()
pd.DataFrame({"timestamp": s.index, "score": sc, "alarm": (sc >= thr).astype(int)}).to_csv(OUT / "scores.csv", index=False)
save_json({"metrics_by_quantile": res.to_dict("index"), "chosen": m}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Tanpa label (kondisi lomba biasa), tetapkan ambang dari kuantil atau kapasitas penanganan alarm.
- Bandingkan metode berbasis musiman (STL/profil mingguan) dengan metode jendela: musiman kuat membuat metode tanpa musiman gagal.
- Gabungkan skor beberapa metode (rata-rata rank) untuk keandalan.